In [5]:
import pandas as pd
from numeric import num, cat

In [6]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

In [7]:
train_numeric = num(train)
test_numeric = num(test)
train_categorical = cat(train)
test_categorical = cat(test)


In [8]:
# ID 기준 병합
train_final = pd.merge(
    train_numeric,
    train_categorical,
    on="ID",
    how="inner"
)

# completed 컬럼을 맨 마지막으로 이동
cols = [c for c in train_final.columns if c != "completed"] + ["completed"]
train_final = train_final[cols]

In [9]:
test_final = pd.merge(
    test_numeric,
    test_categorical,
    on="ID",
    how="inner"
)

In [10]:
train_final.head()

,ID,time_input,class_count,certificate_count,major_data,re_registration,incumbents_level,completed
0,TRAIN_000,2.0,2,6,False,아니요,시니어 (10년차 ~),0
1,TRAIN_001,3.0,1,4,True,아니요,주니어 (0~3년차),0
2,TRAIN_002,10.0,1,4,False,아니요,주니어 (0~3년차),0
3,TRAIN_003,2.0,1,4,False,아니요,주니어 (0~3년차),1
4,TRAIN_004,2.0,1,5,True,아니요,주니어 (0~3년차),0


In [11]:
test_final.head()

,ID,time_input,class_count,certificate_count,major_data,re_registration,incumbents_level
0,TEST_000,2.0,1,6,True,아니요,주니어 (0~3년차)
1,TEST_001,1.0,1,2,True,아니요,시니어 (10년차 ~)
2,TEST_002,5.0,1,3,True,아니요,시니어 (10년차 ~)
3,TEST_003,4.0,2,6,False,아니요,주니어 (0~3년차)
4,TEST_004,1.0,1,3,False,아니요,주니어 (0~3년차)


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings, random
warnings.filterwarnings(action='ignore')

from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler
from category_encoders.ordinal import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold

from sklearn.cluster import KMeans
from catboost import CatBoostClassifier, Pool

train = train_final.copy()
test = test_final.copy()

train.fillna('NaN', inplace=True) 
test.fillna('NaN', inplace=True)

numerical_feats = ['time_input', 'class_count', 'certificate_count']
categorical_feats = ['major_data', 're_registration', 'incumbents_level']

for df in [train,test]:
    df['time_input'] = np.log1p(1+df['time_input'])

In [ ]:
train = train_final.copy()
test = test_final.copy()

In [14]:
train.fillna('NaN', inplace=True) 
test.fillna('NaN', inplace=True)

In [16]:
numerical_feats = ['time_input', 'class_count', 'certificate_count']
categorical_feats = ['major_data', 're_registration', 'incumbents_level']

In [17]:
for df in [train,test]:
    df['time_input'] = np.log1p(1+df['time_input'])

In [19]:
from category_encoders.ordinal import OrdinalEncoder

# 1) ID 처리: 모델에서는 보통 제거 추천
#    만약 꼭 숫자로 바꾸고 싶으면 TRAIN_/TEST_ 떼고 숫자만 남겨
for df in [train, test]:
    if df["ID"].dtype == "object":
        df["ID_num"] = df["ID"].str.extract(r"(\d+)").astype("int64")
    else:
        df["ID_num"] = df["ID"].astype("int64")

# 모델 입력에서는 ID 컬럼은 빼는 걸 추천
drop_cols = ["ID"]  # 원하면 ["ID", "ID_num"] 둘다 빼도 됨

# 2) Ordinal Encoding (그냥 카테고리 -> 정수)
encoder = OrdinalEncoder(cols=categorical_feats, handle_unknown='value', handle_missing='value')

train[categorical_feats] = encoder.fit_transform(train[categorical_feats])
test[categorical_feats]  = encoder.transform(test[categorical_feats])

# 3) X, y 만들기
y = train["completed"].astype(int)
X = train.drop(columns=["completed"] + drop_cols)
X_test = test.drop(columns=drop_cols)

print(X.shape, X_test.shape)


(748, 7) (814, 7)
